# DF-THC quick rank/width probe: (R=27, B=54)

A short GPU-only continuous-factorization experiment. It truncates the public (65,60,30) seed to B=54, runs a short refit, drops to R=27, and refits briefly. It does **not** quantize angles, build QROM circuits, or run the official verifier. The goal is only to decide whether this direction merits a full optimization pass.

In [ ]:
import os, sys, json, time, pathlib, math
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import numpy as np
import torch
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
torch.set_num_threads(1)
torch.backends.cuda.matmul.allow_tf32 = False

INPUT = pathlib.Path("/kaggle/input").resolve()
WORK = pathlib.Path("/kaggle/working").resolve()
SRC = WORK / "src"
RESULTS = WORK / "dfthcsos-rb-probe"
SRC.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

comp_candidates = sorted({p.parent.parent for p in INPUT.rglob("verify_submission_invsqrt.py") if p.parent.name == "invsqrt"})
comps = [p.parent for p in comp_candidates if (p.parent / "kickmix/kickmix.so").is_file()]
if len(comps) != 1:
    raise FileNotFoundError(f"Expected one competition toolbox, found {comps}")
COMP = comps[0]

atlas_candidates = sorted({p.parent for p in INPUT.rglob("atlas_manifest.json")
                           if (p.parent / "fusion/selected_coefficients.npz").is_file()
                           and (p.parent / "selected/dfthcsos/submission.npz").is_file()})
if len(atlas_candidates) != 1:
    raise FileNotFoundError(f"Expected one Quantum Circuit Toolbox dataset, found {atlas_candidates}")
TOOLBOX = atlas_candidates[0]
sys.path.extend([str(COMP / "verifiers"), str(TOOLBOX / "code"), str(SRC)])

print("Competition:", COMP)
print("Toolbox:", TOOLBOX)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("This probe requires a Kaggle GPU; GPU was requested in kernel metadata.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
%%writefile src/chemistry_refine.py
"""Physical Gram-preserving rotations and effective-normalization refinement.
New derivative tooling: Apache-2.0. Retains credit to Kozykappa's public
physical compression/sparse rotations and anazemcev's effective-norm method.
"""
from __future__ import annotations
import importlib,math,sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any,Mapping
import numpy as np
from scipy.optimize import minimize
PARAMETER_KEYS=("U","W","H2_I","H1_I","Hsym_lower")


@dataclass
class Context:
    reference: dict[str, Any]
    fixed_shifts: dict[str, np.ndarray]
    energy_upper: float
    math: Any
    prepare_model: Any
    packed_target: np.ndarray | None = None


def load_context(verifiers: str | Path) -> Context:
    """Read the checksummed official integrals and fixed symmetry shifts."""
    path = Path(verifiers).resolve()
    if path.name == "dfthcsos":
        path = path.parent
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
    reference = importlib.import_module("dfthcsos.scoreinfra.reference")
    maths = importlib.import_module("dfthcsos.scoreinfra.dfthc_math")
    costs = importlib.import_module("dfthcsos.scoreinfra.costs_prepare")
    integrals = reference.load_reference(reference.check_fcidump())
    return Context(integrals, reference.load_fixed_shifts(), reference.UPPER_ENERGY,
                   maths, costs.SpinFreeData_V1)


def load_factorization(path: str | Path) -> dict[str, np.ndarray]:
    with np.load(path, allow_pickle=False) as archive:
        data = {key: np.array(archive[key]) for key in PARAMETER_KEYS}
        for key in ("angle_words", "tree_edges"):
            if key in archive:
                data[key] = np.array(archive[key], dtype=np.int64)
    return data


def copy_data(data: Mapping[str, Any]) -> dict[str, np.ndarray]:
    return {key: np.array(value, copy=True) for key, value in data.items()}


def native_inputs(context: Context, data: Mapping[str, Any]):
    """Use exactly the fixed-shift and float32 symmetrization convention."""
    w = np.asarray(data["W"], dtype=np.float64)
    offset = np.asarray(data["H2_I"], dtype=np.float64) - w.sum(axis=1)
    n = np.shape(data["U"])[-1]
    shift = np.asarray(context.math.symmetric_from_lower(
        context.fixed_shifts["Hsym_lower"], n), dtype=np.float64)
    beta = float(context.fixed_shifts["H1_I"]) - 0.5 * np.trace(shift)
    return w, offset, shift, float(beta)


def encode_vectors(vectors: np.ndarray, tree_edges: np.ndarray, brot: int) -> np.ndarray:
    """Reverse the ordered Givens tree, then round angles to nearest words."""
    if not 1 <= brot <= 30:
        raise ValueError("this numerical encoder supports 1 <= brot <= 30")
    work = np.asarray(vectors, dtype=np.float64).copy()
    work /= np.linalg.norm(work, axis=1, keepdims=True)
    angles = np.empty((len(work), len(tree_edges)), dtype=np.float64)
    for j in reversed(range(len(tree_edges))):
        parent, child = tree_edges[j]
        angles[:, j] = np.arctan2(work[:, child], work[:, parent])
        work[:, parent] = np.hypot(work[:, parent], work[:, child])
        work[:, child] = 0.0
    return np.rint(angles * (2 ** (brot - 1) / np.pi)).astype(np.int64) % (1 << brot)


def decoded_vectors(context: Context, data: Mapping[str, Any], brot: int):
    n = np.shape(data["U"])[-1]
    r, b, _ = np.shape(data["W"])
    vectors = context.math.reconstruct_u_vectors(
        np.asarray(data["angle_words"]) / (1 << (brot - 1)), data["tree_edges"])
    return vectors[:n], vectors[n:].reshape(r, b, n)


def refresh_one_body(context: Context, data: Mapping[str, Any], brot: int):
    """Re-encode the actual h1-prime eigenvectors after modifying factors."""
    result = copy_data(data)
    _, u = decoded_vectors(context, result, brot)
    w, offsets, shift, beta = native_inputs(context, result)
    n = u.shape[-1]
    coefficients = np.einsum("rc,rbc->rb", offsets, w)
    h1 = (context.reference["h1_thc"]
          - 0.5 * context.reference["electrons"] * shift + beta * np.eye(n)
          - np.einsum("rb,rbp,rbq->pq", coefficients, u, u, optimize=True))
    _, vectors = np.linalg.eigh(h1)
    result["angle_words"][:n] = encode_vectors(vectors.T, result["tree_edges"], brot)
    return result


def evaluate_words(context: Context, data: Mapping[str, Any], brot: int):
    w, offsets, shift, beta = native_inputs(context, data)
    return context.math.evaluate_dfthc_angle_words(
        data["angle_words"], w, offsets, shift, beta, context.reference,
        brot=brot, tree_edges=data["tree_edges"], energy_upper=context.energy_upper)


def passes_accuracy(metrics: Mapping[str, Any], *, error_margin: float = 0.0) -> bool:
    return bool(metrics["error_1"] <= 0.020 - error_margin
                and metrics["error_2"] <= 0.020 - error_margin
                and metrics["Lambda"] > 49.0 and metrics["E_gap"] > 3.914898
                and metrics["gap_in_domain"])


def sparsify_columns(data: Mapping[str, Any], *, rounds: int = 20,
                     pairs_per_round: int | None = None, seed: int = 7361):
    """Exact orthogonal rotations reduce sum_c ||[w; w_B]_c||_1 squared.

    All zero-crossing angles are checked for every pair; accepting a rotation
    preserves both the two-body Gram tensor and the absorbed one-body term.
    """
    result = copy_data(data)
    a = np.concatenate([np.asarray(data["W"], dtype=np.float64),
                        np.asarray(data["H2_I"], dtype=np.float64)[:, None, :]], axis=1)
    r, _, c = a.shape
    before = float(np.sum(np.abs(a).sum(axis=1) ** 2))
    original_gram = a @ a.swapaxes(1, 2)
    rng = np.random.default_rng(seed)
    accepted = 0
    if c > 1:
        for _ in range(rounds):
            for _ in range(pairs_per_round or 4 * c):
                i, j = rng.choice(c, 2, replace=False)
                x, y = a[:, :, i].copy(), a[:, :, j].copy()
                theta = np.mod(np.arctan2(y, x), np.pi / 2)
                theta = np.concatenate([np.zeros((r, 1)), theta], axis=1)
                co, si = np.cos(theta)[:, :, None], np.sin(theta)[:, :, None]
                losses = (np.abs(co * x[:, None] + si * y[:, None]).sum(axis=2) ** 2
                          + np.abs(co * y[:, None] - si * x[:, None]).sum(axis=2) ** 2)
                k = np.argmin(losses, axis=1)
                best = theta[np.arange(r), k]
                old = np.abs(x).sum(axis=1) ** 2 + np.abs(y).sum(axis=1) ** 2
                best = np.where(losses[np.arange(r), k] < old - 1e-12, best, 0.0)
                co, si = np.cos(best)[:, None], np.sin(best)[:, None]
                a[:, :, i], a[:, :, j] = co * x + si * y, co * y - si * x
                accepted += int(np.count_nonzero(best))
    drift = float(np.max(np.abs(a @ a.swapaxes(1, 2) - original_gram)))
    if drift > 1e-8:
        raise ArithmeticError(f"orthogonal column-rotation Gram drift {drift}")
    result["W"], result["H2_I"] = a[:, :-1], a[:, -1]
    return result, {"l1_square_before": before,
                    "l1_square_after": float(np.sum(np.abs(a).sum(axis=1) ** 2)),
                    "accepted_rotations": accepted, "gram_max_drift": drift}


def optimize_effective_norm(context: Context, data: Mapping[str, Any], brot: int,
                            *, maxiter: int = 240, gap_margin: float = 0.001):
    """Optimize the actual lambda_eff, including the LCU one-norm derivative.

    Minimizing the SOS gap alone ignores the way identity offsets change
    Lambda. This routine trades both physical quantities using their exact
    eigenvalue derivatives. A penalty enforces a positive acceptance margin;
    reported quantities are recomputed from the real parameters, never clipped.
    """
    result = copy_data(data)
    _, u = decoded_vectors(context, result, brot)
    w, original, shift, beta = native_inputs(context, result)
    r, _, c = w.shape
    n = u.shape[-1]
    constant = (context.reference["h1_thc"]
                - 0.5 * context.reference["electrons"] * shift + beta * np.eye(n))
    energy = context.energy_upper - context.reference["ecore"] + context.reference["electrons"] * beta
    sum_w = w.sum(axis=1)
    l1_w = np.abs(w).sum(axis=1)
    floor = 3.914898 + gap_margin
    def quantities(flat):
        offsets = flat.reshape(r, c)
        combined = np.einsum("rc,rbc->rb", offsets, w)
        h1 = constant - np.einsum("rb,rbp,rbq->pq", combined, u, u, optimize=True)
        values, vectors = np.linalg.eigh(h1)
        sign_h1 = (vectors * np.sign(values)) @ vectors.T
        contractions = np.einsum("rbp,pq,rbq->rb", u, sign_h1, u, optimize=True)
        gap = float(energy + np.abs(values).sum() - values.sum() + 0.5 * np.sum(offsets ** 2))
        w_b = offsets + sum_w
        lam = float(np.abs(values).sum() + 0.25 * np.sum((l1_w + np.abs(w_b)) ** 2))
        grad_gap = offsets + np.einsum("rb,rbc->rc", 1.0 - contractions, w)
        grad_lam = (-np.einsum("rb,rbc->rc", contractions, w)
                    + 0.5 * (l1_w + np.abs(w_b)) * np.sign(w_b))
        return gap, lam, grad_gap, grad_lam
    def objective(flat):
        gap, lam, grad_gap, grad_lam = quantities(flat)
        radicand = gap * (2.0 * lam - gap)
        if radicand <= 0:
            return 1e12 + abs(radicand), np.zeros_like(flat)
        loss = 0.5 * math.log(radicand)
        gradient = ((lam - gap) * grad_gap + gap * grad_lam) / radicand
        if gap < floor:
            loss += 1e5 * (floor - gap) ** 2
            gradient -= 2e5 * (floor - gap) * grad_gap
        if lam < 49.001:
            loss += 1e5 * (49.001 - lam) ** 2
            gradient -= 2e5 * (49.001 - lam) * grad_lam
        return loss, gradient.ravel()
    gap0, lam0, _, _ = quantities(original.ravel())
    optimized = minimize(objective, original.ravel(), jac=True, method="L-BFGS-B",
                         options={"maxiter": maxiter, "maxcor": 12,
                                  "ftol": 1e-13, "gtol": 1e-8, "maxls": 40})
    offsets = optimized.x.reshape(r, c)
    gap1, lam1, _, _ = quantities(offsets.ravel())
    before = math.sqrt(gap0 * (2.0 * lam0 - gap0))
    after = math.sqrt(gap1 * (2.0 * lam1 - gap1))
    accepted = gap1 > 3.914898 and lam1 > 49.0 and after < before
    if accepted:
        result["H2_I"] = offsets + sum_w
        result = refresh_one_body(context, result, brot)
    return result, {"lambda_eff_before": before, "lambda_eff_after": after,
                    "gap_before": gap0, "gap_after": gap1,
                    "Lambda_before": lam0, "Lambda_after": lam1,
                    "iterations": int(optimized.nit), "success": bool(optimized.success),
                    "accepted": bool(accepted)}



In [ ]:
%%writefile src/dfthc_train.py
"""DF-THC group compression for the dfthcsos benchmark (optional GPU retraining).
SPDX-License-Identifier: Apache-2.0

The score's ANGLE lookups cost about one Toffoli per four addresses per circuit,
so the chemistry cost is close to proportional to the number of loaded vectors
N + R*B.  Starting from the public (R, B, C) = (65, 60, 30) factorization this
module:

1. refits the two-body structure sum_rc X^{rc} (x) X^{rc} against the shifted
   target with a penalty on D(w) = 1/2 sum[(sum_b |w|)^2 - (sum_b w)^2], which is
   zero exactly when every square X^{rc} is semidefinite.  With the fixed
   symmetry shift, lambda_eff^2 = E_gap (559.13 + 2 P_pos + excess) and the
   excess is bounded below by D, so keeping D small keeps lambda_eff near its
   floor sqrt(3.9149 * 559.13) = 46.79;
2. drops the least important groups (largest-norm squares kept) with a refit
   after every drop, 65 -> 55 -> 48 -> 42 -> 36 -> 32 -> 28 -> 26.  The last
   two refits are long (150k and 100k Adam steps, learning rate 2e-3 -> 2e-5):
   at R = 28 the error falls from 20.8 mHa after 15k steps to 15.6 mHa;
3. quantizes the two-body Givens angles to brot = 10 bits in the signed
   half-open gauge used by the four-address QROM, freezing the angles with the
   largest Gauss-Newton impact first so the still-free angles and the weights
   absorb the rounding error.

The weights stay continuous (they are not loaded by the circuits), and the
W offsets are refined afterwards by chemistry_refine.  Everything here needs a
CUDA GPU; the notebook loads the result from its attached dataset so that it
runs without one.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch


# ---------------------------------------------------------------------------
# Target: tei + (S x I + I x S)/2 in the orthonormal symmetric pair basis
# ---------------------------------------------------------------------------

class Target:
    def __init__(self, reference, fixed_shifts, symmetric_from_lower, device='cuda'):
        n = int(reference['orbitals'])
        s = np.asarray(symmetric_from_lower(fixed_shifts['Hsym_lower'], n), dtype=np.float64)
        t = np.asarray(reference['tei'], dtype=np.float64).copy()
        d = np.arange(n)
        t[:, :, d, d] += 0.5 * s[:, :, None]
        t[d, d, :, :] += 0.5 * s[None, :, :]
        rows, cols = np.tril_indices(n)
        scale = np.where(rows == cols, 1.0, math.sqrt(2.0))
        pairs = rows * n + cols
        tm = t.reshape(n * n, n * n)
        ts = tm[np.ix_(pairs, pairs)] * np.outer(scale, scale)
        self.n = n
        self.Ts = torch.tensor(ts, device=device, dtype=torch.float32)
        self.ri = torch.tensor(rows, device=device)
        self.ci = torch.tensor(cols, device=device)
        self.scale = torch.tensor(scale, device=device, dtype=torch.float32)


def x_sym(u, w, tg):
    """Rows of the R*C squares X^{rc} = sum_b w_b u_b u_b^T in the symmetric pair basis."""
    v = u[:, :, tg.ri] * u[:, :, tg.ci] * tg.scale
    a = torch.einsum('rbm,rbc->rcm', v, w)
    return a.reshape(-1, a.shape[-1])


def frob_err(a, tg):
    e = a.T @ a - tg.Ts
    return torch.sqrt((e * e).sum())


def d_penalty(w):
    a = w.abs().sum(dim=1)
    s = w.sum(dim=1)
    return 0.5 * (a * a - s * s).sum()


# ---------------------------------------------------------------------------
# Log-depth Givens tree (the competition's starter tree), level-parallel
# ---------------------------------------------------------------------------

def log_depth_tree_edges(n):
    edges, level, depth = [], [0], 0
    while len(edges) < n - 1:
        nxt = []
        for label in level:
            nxt.append(label)
            right = label + (1 << depth)
            if right < n:
                edges.append((label, right))
                nxt.append(right)
        level, depth = nxt, depth + 1
    return np.asarray(edges, dtype=np.int64)


EDGES = log_depth_tree_edges(76)


class Tree:
    def __init__(self, edges=EDGES, n=76, device='cuda'):
        self.n = n
        levels, cur, used = [], [], set()
        for t, (p, q) in enumerate(edges):
            if p in used or q in used:
                levels.append(cur)
                cur, used = [], set()
            cur.append(t)
            used.update((int(p), int(q)))
        levels.append(cur)
        self.plan = []
        for level in levels:
            colmap = np.full(n, n - 1, dtype=np.int64)
            sign = np.zeros(n)
            perm = np.arange(n)
            for t in level:
                p, q = int(edges[t][0]), int(edges[t][1])
                colmap[p] = colmap[q] = t
                sign[p], sign[q] = -1.0, 1.0
                perm[p], perm[q] = q, p
            self.plan.append((torch.tensor(colmap, device=device), torch.tensor(sign, device=device, dtype=torch.float32),
                              torch.tensor(perm, device=device)))

    def __call__(self, theta):
        k = theta.shape[0]
        ang = math.pi * theta
        c = torch.cat([torch.cos(ang), torch.ones(k, 1, device=theta.device)], dim=1)
        s = torch.cat([torch.sin(ang), torch.zeros(k, 1, device=theta.device)], dim=1)
        u = torch.zeros(k, self.n, device=theta.device, dtype=theta.dtype)
        u[:, 0] = 1.0
        for colmap, sign, perm in self.plan:
            u = u * c[:, colmap] + u[:, perm] * (s[:, colmap] * sign)
        return u


def gauge_angles(u, edges=EDGES):
    """Signed half-open gauge: theta/pi in [-1/2, 1/2], each row reproduced up to sign."""
    a = np.array(u, dtype=np.float64, copy=True)
    ang = np.zeros((a.shape[0], len(edges)))
    for j in reversed(range(len(edges))):
        p, q = edges[j]
        norm = np.hypot(a[:, p], a[:, q])
        sign = np.where(np.abs(a[:, p]) < 1e-12, np.where(a[:, q] > 0, -1.0, 1.0), np.where(a[:, p] < 0, -1.0, 1.0))
        ang[:, j] = np.where(norm < 1e-12, 0.0, np.arctan2(a[:, q] * sign, a[:, p] * sign))
        a[:, p] = np.where(norm < 1e-12, 0.0, norm * sign)
        a[:, q] = 0.0
    return ang / np.pi


# ---------------------------------------------------------------------------
# Stages
# ---------------------------------------------------------------------------

def fit_structure(U, w, tg, steps=15000, lr0=1e-3, lr1=1e-4, kappa=2e-5, log=print):
    """Adam on (U, w) for ||sum X(x)X - T||_F + kappa D(w); returns the best (error, U, w)."""
    def loss(U, w):
        u = U / U.norm(dim=2, keepdim=True)
        return frob_err(x_sym(u, w, tg), tg)
    lf = loss
    U = U.clone().requires_grad_(True)
    w = w.clone().requires_grad_(True)
    opt = torch.optim.Adam([U, w], lr=lr0)
    gamma = (lr1 / lr0) ** (1.0 / steps)
    best = (float('inf'), None, None)
    t0 = time.time()
    for s in range(steps + 1):
        e = lf(U, w)
        if s % 50 == 0 and e.item() < best[0]:
            best = (e.item(), U.detach().clone(), w.detach().clone())
        if s % 250 == 0:
            log(f'  fit step {s:6d} error_2 {1e3 * e.item():8.3f} mHa  D {d_penalty(w).item():7.2f}  {time.time() - t0:5.0f}s')
        if s == steps:
            break
        opt.zero_grad(set_to_none=True)
        (e + kappa * d_penalty(w)).backward()
        opt.step()
        for g in opt.param_groups:
            g['lr'] *= gamma
    return best


def drop_groups(U, w, keep):
    with torch.no_grad():
        u = U / U.norm(dim=2, keepdim=True)
        x = torch.einsum('rbp,rbq,rbc->rcpq', u, u, w)
        idx = torch.argsort(-(x * x).sum(dim=(1, 2, 3)))[:keep].sort().values
    return U[idx].contiguous(), w[idx].contiguous()


def quantize(U, w, tg, brot=10, kappa=2e-5, steps=1200, final_steps=4000, probes=16, log=print,
             fracs=(0.3, 0.5, 0.65, 0.75, 0.83, 0.89, 0.93, 0.95, 0.97, 0.98, 0.985, 0.99, 0.993, 0.996, 0.998, 1.0)):
    """Progressive, sensitivity-ordered quantization of the two-body gauge angles."""
    R, B, N = U.shape
    tree = Tree(device=U.device)
    u0 = (U / U.norm(dim=2, keepdim=True)).detach().cpu().numpy().reshape(-1, N)
    scale = float(1 << (brot - 1))
    lo, hi = -0.5, 0.5 - 1.0 / scale
    theta = torch.tensor(gauge_angles(u0), device=U.device, dtype=torch.float32)
    w = w.detach().clone()
    frozen = torch.zeros_like(theta, dtype=torch.bool)
    qvals = torch.zeros_like(theta)

    def qround(t):
        return torch.clamp(torch.round(t * scale) / scale, lo, hi)

    def resid(th, w):
        return frob_err(x_sym(tree(th).reshape(R, B, N), w, tg), tg)
    lf = resid

    def impact():
        th = torch.where(frozen, qvals, theta).detach().clone().requires_grad_(True)
        a = x_sym(tree(th).reshape(R, B, N), w, tg)
        e = a.T @ a - tg.Ts
        acc = torch.zeros_like(th)
        for _ in range(probes):
            v = torch.randn_like(e)
            g, = torch.autograd.grad((e * (v + v.T)).sum(), th, retain_graph=True)
            acc += g * g
        r = qround(theta) - theta
        return acc * r * r

    for stage, frac in enumerate(list(fracs) + [None]):
        if frac is not None:
            need = int(frac * theta.numel()) - int(frozen.sum())
            if need > 0:
                imp = impact()
                with torch.no_grad():
                    imp[frozen] = -1.0
                    idx = torch.argsort(imp.flatten(), descending=True)[:need]
                    fl, qv = frozen.flatten().clone(), qvals.flatten().clone()
                    fl[idx] = True
                    qv[idx] = qround(theta).flatten()[idx]
                    frozen, qvals = fl.view_as(frozen), qv.view_as(qvals)
        n_steps = steps if frac is not None else final_steps
        th_p = theta.clone().requires_grad_(True)
        w_p = w.clone().requires_grad_(True)
        groups = [{'params': [w_p], 'lr': 3e-4}] + ([] if bool(frozen.all()) else [{'params': [th_p], 'lr': 1e-3}])
        opt = torch.optim.Adam(groups)
        ff = frozen.float()
        best = (float('inf'), None, None, None)
        for s in range(n_steps + 1):
            e = lf(ff * qvals + (1 - ff) * th_p, w_p)
            total = e + kappa * d_penalty(w_p)
            if s % 25 == 0 and total.item() < best[0]:
                best = (total.item(), th_p.detach().clone(), w_p.detach().clone(), e.item())
            if s == n_steps:
                break
            opt.zero_grad(set_to_none=True)
            total.backward()
            opt.step()
            with torch.no_grad():
                th_p.clamp_(lo, hi)
            for g in opt.param_groups:
                g['lr'] *= 0.2 ** (1.0 / n_steps)
        theta, w = best[1], best[2]
        log(f'  quantize stage {stage:2d} frozen {float(frozen.float().mean()):.4f} error_2 {1e3 * best[3]:8.3f} mHa')
    final = torch.where(frozen, qvals, qround(theta)).cpu().numpy().astype(np.float64)
    words = np.mod(np.round(final * scale).astype(np.int64), 1 << brot)
    return words, w.cpu().numpy().astype(np.float64)


def compress(public_npz, reference, fixed_shifts, symmetric_from_lower, groups=(55, 48, 42, 36, 32, 28, 26),
             steps=15000, long_fits=((28, 150000, True), (26, 100000, False)), kappa=2e-5, log=print):
    """Full retraining pipeline; returns (two-body angle words, weights w).

    long_fits lists (R, steps, after_short_fit): the refit after dropping to R
    is a long one (learning rate 2e-3 -> 2e-5), run after the usual short refit
    when after_short_fit is set and instead of it otherwise.
    """
    torch.backends.cuda.matmul.allow_tf32 = False
    tg = Target(reference, fixed_shifts, symmetric_from_lower)
    z = np.load(public_npz)
    U = torch.tensor(z['U'], device='cuda', dtype=torch.float32)
    w = torch.tensor(z['W'], device='cuda', dtype=torch.float32)
    log(f'start {tuple(U.shape[:2]) + (w.shape[2],)}')
    err, U, w = fit_structure(U, w, tg, steps=steps, kappa=kappa, log=log)
    long = {r: (n, after) for r, n, after in long_fits}
    for keep in groups:
        U, w = drop_groups(U, w, keep)
        if keep not in long or long[keep][1]:
            err, U, w = fit_structure(U, w, tg, steps=steps, kappa=kappa, log=log)
        if keep in long:
            err, U, w = fit_structure(U, w, tg, steps=long[keep][0], lr0=2e-3, lr1=2e-5, kappa=kappa, log=log)
        log(f'R={keep}: continuous error_2 {1e3 * err:.3f} mHa')
    return quantize(U, w, tg, kappa=kappa, log=log)


In [ ]:
import numpy as np
import torch
import time
import json
import chemistry_refine as C
import dfthc_train as T

started = time.monotonic()
ctx = C.load_context(COMP / "verifiers")

# Current compressed factorization: official metrics, for a comparable baseline.
factor_path = next(INPUT.rglob("dfthc_group_v2_26_60_30.npz"))
with np.load(factor_path, allow_pickle=False) as z:
    packed = {k: np.array(z[k]) for k in z.files}
words = packed["angle_words"].astype(np.int64)
edges = packed["tree_edges"].astype(np.int64)
w_base = np.asarray(packed["W"], dtype=np.float64)
h2_base = np.asarray(packed["H2_I"], dtype=np.float64)
n_orb = int(ctx.reference["orbitals"])
decoded = ctx.math.reconstruct_u_vectors(words / (1 << 9), edges)
r_base, b_base, _ = w_base.shape
baseline = {
    "U": decoded[n_orb:].reshape(r_base, b_base, n_orb),
    "W": w_base,
    "H2_I": h2_base,
    "H1_I": np.asarray(ctx.fixed_shifts["H1_I"]),
    "Hsym_lower": np.asarray(ctx.fixed_shifts["Hsym_lower"]),
    "tree_edges": edges,
    "angle_words": words,
}
base_metrics = C.evaluate_words(ctx, baseline, 10)
print("BASELINE (official metric, current R=26 B=60):",
      json.dumps({k: float(base_metrics[k]) for k in ("error_1", "error_2", "lambda_eff", "E_gap")}), flush=True)

# Candidate: reduce B=60 to B=54, then follow the original staged group-drop idea.
public_path = TOOLBOX / "selected/dfthcsos/submission.npz"
with np.load(public_path, allow_pickle=False) as z:
    U0 = np.asarray(z["U"], dtype=np.float32)[:, :54, :].copy()
    W0 = np.asarray(z["W"], dtype=np.float32)[:, :54, :].copy()
print("Public seed after B truncation:", U0.shape, W0.shape, flush=True)

target = T.Target(ctx.reference, ctx.fixed_shifts, ctx.math.symmetric_from_lower, device="cuda")
def continuous_error_mha(U, W):
    ut = torch.as_tensor(U, device="cuda", dtype=torch.float32)
    wt = torch.as_tensor(W, device="cuda", dtype=torch.float32)
    ut = ut / ut.norm(dim=2, keepdim=True)
    return 1000.0 * float(T.frob_err(T.x_sym(ut, wt, target), target).item())

truncated_error = continuous_error_mha(U0, W0)
print(f"TRUNCATED ONLY: continuous error_2 proxy = {truncated_error:.4f} mHa", flush=True)

U = torch.as_tensor(U0, device="cuda", dtype=torch.float32)
W = torch.as_tensor(W0, device="cuda", dtype=torch.float32)
print("INITIAL SHORT REFIT: B=54, all 65 groups, 1500 steps", flush=True)
err, U, W = T.fit_structure(U, W, target, steps=1500, lr0=1e-3, lr1=2e-4,
                             kappa=2e-5, log=print)
initial_error = 1000 * float(err)
print(f"After initial refit: {initial_error:.4f} mHa", flush=True)

# The previous direct-drop screen used steps=3000 after dropping straight to R=27.
# This revised run preserves the staged path: 65 -> 55 -> 48 -> 42 -> 36 -> 32 -> 28 -> 27.
stage_errors = []
for keep in (55, 48, 42, 36, 32, 28, 27):
    U, W = T.drop_groups(U, W, keep)
    print(f"STAGED REFIT: R={keep}, B=54, 1000 steps", flush=True)
    err, U, W = T.fit_structure(U, W, target, steps=1000, lr0=1e-3, lr1=2e-4,
                                 kappa=2e-5, log=print)
    row = {"R": keep, "B": 54, "steps": 1000, "error_2_mHa": 1000*float(err)}
    stage_errors.append(row)
    print("STAGE RESULT:", json.dumps(row), flush=True)

candidate_error = continuous_error_mha(U.detach().cpu().numpy(), W.detach().cpu().numpy())
candidate_vectors = n_orb + 27 * 54
baseline_vectors = n_orb + 26 * 60
summary = {
    "status": "SCREEN_ONLY_NOT_OFFICIALLY_VERIFIED",
    "baseline": {"R": 26, "B": 60, "vectors_N_plus_RB": int(baseline_vectors),
                 "official_error_1_mHa": 1000*float(base_metrics["error_1"]),
                 "official_error_2_mHa": 1000*float(base_metrics["error_2"]),
                 "lambda_eff": float(base_metrics["lambda_eff"])},
    "candidate": {"R": 27, "B": 54, "vectors_N_plus_RB": int(candidate_vectors),
                  "vector_count_reduction_percent": 100*(1-candidate_vectors/baseline_vectors),
                  "truncated_seed_continuous_error_2_mHa": float(truncated_error),
                  "after_initial_refit_error_2_mHa": float(initial_error),
                  "stage_errors": stage_errors,
                  "final_continuous_error_2_mHa": float(candidate_error),
                  "under_20_mHa_continuous_screen": bool(candidate_error <= 20.0)},
    "steps": {"B54_all_groups": 1500, "each_staged_group_drop": 1000,
              "staged_keeps": [55, 48, 42, 36, 32, 28, 27]},
    "elapsed_seconds": round(time.monotonic()-started, 2),
    "limitations": "Short GPU continuous-factorization screen only. No angle quantization, QROM build, circuit resource count, or official verifier run."
}
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
np.savez_compressed(RESULTS / "candidate_R27_B54_continuous.npz",
                    U=U.detach().cpu().numpy(), W=W.detach().cpu().numpy())
print("FINAL SUMMARY\n" + json.dumps(summary, indent=2), flush=True)
